## Column Transformer
 - It is a ml technique which combine multiple preprocessing technique  and execute  parallely.
 - It column transformer only we use preprocessing technique  not an algorithm
 - we don't want to create multiple objects


        - syntax :  from sklearn.compose import columnTransformer

In [ ]:
# 1. Load dataset

# 2. Separate features and target
X = df.drop("Target", axis=1)
y = df["Target"]

# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 4. Create ColumnTransformer
preprocessor = ColumnTransformer(...)

# 5. Create Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", YourModel())
])

# 6. Train
pipeline.fit(X_train, y_train)

# 7. Predict
y_pred = pipeline.predict(X_test)

## same  KNN but using columnTransformer and Pipeline

In [15]:
# 1. Load dataset
import pandas as pd
df=pd.read_csv("retail_customer_segmentation.csv")
df.head()

,customer_id,age,annual_income,months_active,avg_monthly_spend,purchase_frequency,avg_order_value,discount_usage_rate,return_rate,browsing_time_minutes,support_interactions,payment_method,region,customer_segment
0,33554,53,100473.211709,63,121.430565,0.817268,66.820403,0.117256,0.023144,77.298393,2.0,Card,Semi-Urban,Occasional
1,9428,54,54730.644845,67,572.552674,3.176551,137.087449,0.261647,0.429054,92.132565,2.0,Wallet,Urban,Occasional
2,200,44,58268.121079,57,266.593896,2.713168,71.796888,0.284785,0.011854,155.194768,1.0,UPI,Rural,Occasional
3,12448,54,64829.795654,40,691.452358,5.553977,105.501185,0.104832,0.399686,113.917756,0.0,Wallet,Rural,High_Value
4,39490,28,27431.467873,15,832.664792,1.348389,354.568534,0.409204,0.039517,50.123656,1.0,Card,Semi-Urban,Occasional


In [17]:
#check null values
df.isnull().sum()

customer_id                 0
age                         0
annual_income            3075
months_active               0
avg_monthly_spend        2520
purchase_frequency       1979
avg_order_value             0
discount_usage_rate      2549
return_rate              2487
browsing_time_minutes    3934
support_interactions     1988
payment_method              0
region                      0
customer_segment            0
dtype: int64

In [20]:
#this for single column
df["annual_income"]=df["annual_income"].fillna(df["annual_income"].median())

In [22]:
df.isnull().sum()

customer_id                 0
age                         0
annual_income               0
months_active               0
avg_monthly_spend        2520
purchase_frequency       1979
avg_order_value             0
discount_usage_rate      2549
return_rate              2487
browsing_time_minutes    3934
support_interactions     1988
payment_method              0
region                      0
customer_segment            0
dtype: int64

In [26]:
#for multiple columns 
df.fillna(df.median(numeric_only=True),inplace=True)

In [28]:
df.isnull().sum()

customer_id              0
age                      0
annual_income            0
months_active            0
avg_monthly_spend        0
purchase_frequency       0
avg_order_value          0
discount_usage_rate      0
return_rate              0
browsing_time_minutes    0
support_interactions     0
payment_method           0
region                   0
customer_segment         0
dtype: int64

## we get no now  we dont have null values

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customer_id            50000 non-null  int64  
 1   age                    50000 non-null  int64  
 2   annual_income          50000 non-null  float64
 3   months_active          50000 non-null  int64  
 4   avg_monthly_spend      50000 non-null  float64
 5   purchase_frequency     50000 non-null  float64
 6   avg_order_value        50000 non-null  float64
 7   discount_usage_rate    50000 non-null  float64
 8   return_rate            50000 non-null  float64
 9   browsing_time_minutes  50000 non-null  float64
 10  support_interactions   50000 non-null  float64
 11  payment_method         50000 non-null  object 
 12  region                 50000 non-null  object 
 13  customer_segment       50000 non-null  object 
dtypes: float64(8), int64(3), object(3)
memory usage: 5.3+ 

In [59]:
# 2. Separate features and target
x=df.drop("customer_segment",axis=1)
y=df.customer_segment
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)


In [61]:
# 3. Split data
from sklearn.model_selection import train_test_split

xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [63]:
# 4. Create ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
num_cols=x.select_dtypes(include=["int64", "float64"]).columns

preprocessing=ColumnTransformer(
        transformers=[("encoder",OneHotEncoder(handle_unknown="ignore"),["payment_method","region"]),
                    ("scaler",StandardScaler(),num_cols)]
)

In [65]:
# 5. Create Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessing),
    ("model", KNeighborsClassifier(n_neighbors=3))
])


In [67]:
# Train
pipeline.fit(xtrain, ytrain)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int32](4,)","[0,1,2,3]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['customer_id','age','annual_income',...,'support_interactions', 'payment_method','region']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('scaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [69]:
# Predict
y_pred = pipeline.predict(xtest)

In [80]:
#Model evalution for test data
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print("Acuuracy scare:",accuracy_score(ytest,y_pred))
print(confusion_matrix(ytest,y_pred))

Acuuracy scare: 0.5859
[[ 622  246  106   75]
 [ 294 1127  166  298]
 [ 161  350 2991  845]
 [ 158  490  952 1119]]


In [84]:
print(classification_report(ytest,y_pred))

              precision    recall  f1-score   support

           0       0.50      0.59      0.54      1049
           1       0.51      0.60      0.55      1885
           2       0.71      0.69      0.70      4347
           3       0.48      0.41      0.44      2719

    accuracy                           0.59     10000
   macro avg       0.55      0.57      0.56     10000
weighted avg       0.59      0.59      0.58     10000

